# 05 - Collect all results
Prints the key tables from the earlier notebooks and zips everything (tables, figures, config) so you can download it. Run it after 01-04.

In [1]:
# ---- 1. connect Drive, install, load the code ----
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Not running in Colab:', e)
!pip -q install pydicom
import sys, os, numpy as np, pandas as pd, matplotlib.pyplot as plt
CODE_DIR = '/content/drive/MyDrive/mammo_semcom'          # <-- the folder you uploaded (contains src/ and notebooks/)
sys.path.insert(0, CODE_DIR)
from src.config import Cfg

cfg = Cfg(
    cbis_root = '/content/drive/MyDrive/CBIS-DDSM',        # raw data folder: NOT needed after notebook 00 (the cache is used)
    work_dir  = '/content/drive/MyDrive/mammo_semcom_work', # <-- all caches, models, results and figures go here
    max_images = 0,                                         # 0 = everything; try 150 first for a quick dry run
).ensure()
cfg.save()
print(cfg.work_dir)

Mounted at /content/drive
/content/drive/MyDrive/mammo_semcom_work


In [2]:
import glob, subprocess
def show(name, cols=None, query=None):
    p = cfg.path('results', name)
    if not os.path.exists(p): print(f'[missing] {name} - run the notebook that produces it'); return
    d = pd.read_csv(p)
    if query: d = d.query(query)
    print('\n=== ' + name); print((d[cols] if cols else d).round(3).to_string(index=False))
show('baseline.csv')
show('transmission_summary.csv', ['receiver', 'scheme', 'source', 'rate', 'AUC', 'sens_at_spec', 'loc_hit', 'lesion_loss'], 'rate in [0.0, 0.3, 0.5]')
show('transmission_stats.csv', query="kind == 'AUC difference'")
show('uncertainty_quality.csv'); show('uncertainty_under_loss.csv'); show('adaptive_loop.csv')
show('active_learning_summary.csv'); show('active_learning_alc_gain.csv')
print('\nfigures:'); [print('  ', os.path.basename(f)) for f in sorted(glob.glob(cfg.path('figures', '*.png')))]


=== baseline.csv
                                stage                model  n   AUC  ci_low  ci_high  sens_at_90spec  pAUC_FPR10  localization_hit_rate
 classifier only (true lesion centre) MC-dropout (1 model) 23 0.598   0.357    0.833           0.167       0.482                    NaN
 classifier only (true lesion centre)  ensemble (3 models) 23 0.606   0.357    0.844           0.167       0.482                    NaN
end-to-end (localize, crop, classify) MC-dropout (1 model) 22 0.777   0.554    0.949           0.273       0.617                  0.273
end-to-end (localize, crop, classify)  ensemble (3 models) 22 0.851   0.653    0.983           0.545       0.587                  0.273

=== transmission_summary.csv
receiver   scheme source  rate   AUC  sens_at_spec  loc_hit  lesion_loss
  robust    burst   none   0.3 0.466         0.091    0.318        0.361
  robust    burst   none   0.5 0.532         0.000    0.242        0.524
  robust    clean   none   0.0 0.587         0.091   

[None, None, None, None, None, None, None, None]

In [3]:
zip_path = '/content/mammo_semcom_results.zip'
subprocess.run(['zip', '-qr', zip_path, 'results', 'figures', 'config.json'], cwd=cfg.work_dir)
print('saved', zip_path)
try:
    from google.colab import files; files.download(zip_path)
except Exception as e:
    print('Not in Colab:', e)

saved /content/mammo_semcom_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Write-up checklist
1. Data: CBIS-DDSM (cite: Lee et al., 2017, Sci Data; TCIA doi 10.7937/K9/TCIA.2016.7O02S9CY), patient-level split, preprocessing, working resolution.
2. Baseline: classifier-only and end-to-end AUC with bootstrap CI.
3. RQ1: transmission figure + stats table (semantic/pred vs uniform and burst; oracle gap; robust receiver).
4. RQ2: calibration, error detection, risk-coverage, adaptive loop (accuracy gain vs extra bandwidth vs referral rate).
5. RQ3: active-learning curves, ALC table (report a null result honestly if that is what you see).
6. Limitations: simulated channel, simulated radiologist, digitized-film dataset, working resolution below native, single dataset (add external validation if you can obtain VinDr-Mammo).
7. Not a medical device; research prototype.